# US Accidents — Production ML Pipeline v3 (No Data Leakage)

Fixed version with:
- Removed Duration_Minutes (DATA LEAKAGE)
- Using only pre-incident features
- Location_Risk_Score included
- Better feature selection
- Severity threshold analysis
- Honest model evaluation

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report, confusion_matrix,
    f1_score, precision_recall_curve, brier_score_loss
)
from sklearn.calibration import CalibratedClassifierCV
from catboost import CatBoostClassifier

sns.set_style('darkgrid')

In [ ]:
# Load data
df = pd.read_csv(r"F:\Depi Project\Final\Gold_Layer\US_Accidents_Gold_Modeling.csv")

print("Raw dataset shape:", df.shape)
print("\nSeverity distribution:")
print(df['Severity'].value_counts().sort_index())
print(f"\nSeverity >= 3: {(df['Severity'] >= 3).sum()} ({100*(df['Severity'] >= 3).sum()/len(df):.1f}%)")

In [ ]:
# CRITICAL: Remove Duration_Minutes - it's data leakage
# Duration is measured AFTER incident happens, can't use to predict severity

print("\nDATA LEAKAGE CHECK:")
print(f"Original columns: {len(df.columns)}")
print(f"Contains 'Duration_Minutes': {'Duration_Minutes' in df.columns}")

# List of PRE-INCIDENT features only
PRE_INCIDENT_FEATURES = [
    'Temperature_F', 'Humidity_pct', 'Pressure_in', 'Visibility_mi',
    'Wind_Speed_mph', 'Precipitation_in', 'Wind_Chill_Effect',
    'Hour', 'Day_of_Week', 'Month', 'Week_of_Year', 'Year',
    'Is_Weekend', 'Is_Rush_Hour', 'Time_of_Day_Encoded', 'Timezone_Encoded',
    'Is_Extreme_Weather', 'Is_Low_Visibility',
    'Is_Raining', 'Is_Snowing', 'Is_Foggy',
    'Road_Features_Count', 'Has_Traffic_Signal', 'Has_Crossing', 'Has_Junction',
    'Location_Risk_Score',
    'Distance_mi',
    'Temperature_Category_Encoded', 'Sunrise_Sunset_Encoded',
    'Distance_Category_Encoded', 'Weather_Condition_Encoded',
    'Wind_Direction_Encoded'
]

# Verify no leakage
LEAKAGE_FEATURES = ['Duration_Minutes', 'Date_ID', 'Severity_ID', 'Source_ID']
for feat in LEAKAGE_FEATURES:
    if feat in PRE_INCIDENT_FEATURES:
        PRE_INCIDENT_FEATURES.remove(feat)
        print(f"Removed leakage feature: {feat}")

# Keep only features that exist
PRE_INCIDENT_FEATURES = [f for f in PRE_INCIDENT_FEATURES if f in df.columns]

print(f"\nPre-incident features (no leakage): {len(PRE_INCIDENT_FEATURES)}")
print(f"Duration_Minutes excluded: {'Duration_Minutes' not in PRE_INCIDENT_FEATURES}")

In [ ]:
# Create binary target
df['Severity_Binary'] = (df['Severity'] >= 3).astype(int)

print("Target Distribution:")
print(f"  Low (Severity < 3):  {(df['Severity'] < 3).sum():7d} ({100*(df['Severity'] < 3).sum()/len(df):5.1f}%)")
print(f"  High (Severity >= 3): {(df['Severity'] >= 3).sum():7d} ({100*(df['Severity'] >= 3).sum()/len(df):5.1f}%)")

In [ ]:
# Stratified split: 70/15/15
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df['Severity_Binary']
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['Severity_Binary']
)

print("Data split:")
print(f"  Train: {train_df.shape[0]:7d} ({100*train_df.shape[0]/len(df):5.1f}%)")
print(f"  Valid: {valid_df.shape[0]:7d} ({100*valid_df.shape[0]/len(df):5.1f}%)")
print(f"  Test:  {test_df.shape[0]:7d} ({100*test_df.shape[0]/len(df):5.1f}%)")

In [ ]:
# Prepare features
train_median = train_df[PRE_INCIDENT_FEATURES].median(numeric_only=True)

X_train = train_df[PRE_INCIDENT_FEATURES].fillna(train_median)
X_valid = valid_df[PRE_INCIDENT_FEATURES].fillna(train_median)
X_test = test_df[PRE_INCIDENT_FEATURES].fillna(train_median)

y_train = train_df['Severity_Binary'].values
y_valid = valid_df['Severity_Binary'].values
y_test = test_df['Severity_Binary'].values

print(f"Feature matrix shape: {X_train.shape}")
print(f"Features: {len(PRE_INCIDENT_FEATURES)}")

In [ ]:
# Train CatBoost with scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"Class weights:")
print(f"  Negative: {neg_count:7d}")
print(f"  Positive: {pos_count:7d}")
print(f"  Scale pos weight: {scale_pos_weight:.2f}")

model = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    eval_metric='AUC',
    scale_pos_weight=scale_pos_weight,
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    subsample=0.8,
    colsample_bylevel=0.8
)

model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    verbose=False
)

print("\nModel training complete")

In [ ]:
# Calibrate probabilities
valid_proba_uncal = model.predict_proba(X_valid)[:, 1]

calibrator = CalibratedClassifierCV(
    model,
    method='sigmoid',
    cv='prefit'
)

calibrator.fit(X_valid, y_valid)

valid_proba_cal = calibrator.predict_proba(X_valid)[:, 1]

print(f"Calibration metrics:")
print(f"  Uncalibrated Brier: {brier_score_loss(y_valid, valid_proba_uncal):.4f}")
print(f"  Calibrated Brier:   {brier_score_loss(y_valid, valid_proba_cal):.4f}")

In [ ]:
# Optimal threshold using Youden's J
fpr, tpr, thresholds = roc_curve(y_valid, valid_proba_cal)
j_scores = tpr - fpr
best_idx = np.argmax(j_scores)
best_threshold = thresholds[best_idx]

print(f"Optimal threshold (Youden's J): {best_threshold:.4f}")

# Predictions at optimal threshold
test_proba = calibrator.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print(f"\nTest set predictions at threshold {best_threshold:.4f}:")
print(f"  Predicted positives: {test_pred.sum():7d} ({100*test_pred.sum()/len(test_pred):5.1f}%)")
print(f"  Actual positives:    {y_test.sum():7d} ({100*y_test.sum()/len(y_test):5.1f}%)")

In [ ]:
# Test set evaluation
test_auc = roc_auc_score(y_test, test_proba)
test_f1 = f1_score(y_test, test_pred)

print("\n" + "="*70)
print("TEST SET PERFORMANCE (No Data Leakage)")
print("="*70)
print(f"\nROC-AUC: {test_auc:.4f}")
print(f"F1-Score: {test_f1:.4f}")
print()
print(classification_report(
    y_test, test_pred,
    target_names=['Low Severity', 'High Severity']
))

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'Feature': PRE_INCIDENT_FEATURES,
    'Importance': model.get_feature_importance()
}).sort_values('Importance', ascending=False)

print("\nTop 15 Feature Importance:")
print(importance.head(15).to_string())

plt.figure(figsize=(10, 8))
sns.barplot(data=importance.head(15), x='Importance', y='Feature')
plt.title('Top 15 Features (No Leakage)')
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, test_proba)

plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {test_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Test Set (No Data Leakage)')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'],
            yticklabels=['Low', 'High'])
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# 5-fold Cross-validation
X_combined = pd.concat([X_train, X_valid], ignore_index=True)
y_combined = np.concatenate([y_train, y_valid])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_aucs = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_combined, y_combined)):
    X_cv_train, X_cv_val = X_combined.iloc[train_idx], X_combined.iloc[val_idx]
    y_cv_train, y_cv_val = y_combined[train_idx], y_combined[val_idx]
    
    neg_cv = (y_cv_train == 0).sum()
    pos_cv = (y_cv_train == 1).sum()
    
    cv_model = CatBoostClassifier(
        iterations=800, depth=6, learning_rate=0.05,
        loss_function='Logloss', eval_metric='AUC',
        scale_pos_weight=neg_cv / pos_cv,
        random_seed=42, verbose=False, early_stopping_rounds=50
    )
    
    cv_model.fit(X_cv_train, y_cv_train, eval_set=(X_cv_val, y_cv_val), verbose=False)
    cv_auc = roc_auc_score(y_cv_val, cv_model.predict_proba(X_cv_val)[:, 1])
    cv_aucs.append(cv_auc)
    print(f"Fold {fold+1}: AUC = {cv_auc:.4f}")

print(f"\nMean CV AUC: {np.mean(cv_aucs):.4f} (±{np.std(cv_aucs):.4f})")

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY - V3 (No Data Leakage)")
print("="*70)

print(f"""
KEY CHANGES FROM V2:
  - Removed Duration_Minutes (DATA LEAKAGE)
  - Using only pre-incident features
  - Included Location_Risk_Score
  - Using better weather flags

RESULTS:
  Cross-Validation AUC: {np.mean(cv_aucs):.4f} +/- {np.std(cv_aucs):.4f}
  Test Set AUC: {test_auc:.4f}
  Test Set F1: {test_f1:.4f}
  Threshold: {best_threshold:.4f}

INTERPRETATION:
  The model has moderate performance (AUC ~0.60-0.63) because:
  1. Features have weak correlation with severity (~0.11 max)
  2. Severity determined by factors NOT in dataset
  3. Random variation in incident classification

RECOMMENDATIONS:
  1. Verify target definition (Severity >= 3 appropriate?)
  2. Add incident type/description features if available
  3. Add traffic/road characteristics (lanes, speed limit, etc.)
  4. Consider if some severity classifications are subjective
""")